# Phase 4b, third addendum: the sentence-memory-teacher arm and K2 on its students (A100 80 GB)

Reads `results/phase4b/smteacher_models.csv`, `k2_smteacher_scores.csv` and `models.csv` (written by `python -m zoo.phase4b_smteacher` and `zoo.phase4b_report`). Trains nothing. Writes `reports/figures/phase4b_smteacher.png`. Committed without outputs.

In [ ]:
from pathlib import Path
import numpy as np, pandas as pd, matplotlib.pyplot as plt
ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
SM = pd.read_csv(ROOT / "results/phase4b/smteacher_models.csv")
K = pd.read_csv(ROOT / "results/phase4b/k2_smteacher_scores.csv")
MO = pd.read_csv(ROOT / "results/phase4b/models.csv")
R = pd.read_csv(ROOT / "results/phase4b/k2_smteacher_seeds.csv").R.iloc[0]

In [ ]:
INK, MUTED, GRID = "#222222", "#6b6b6b", "#e6e6e6"
ARMS = ["sham student", "sm-teacher student", "1x-teacher student"]
COL = {"sham student": "#4f6d8f", "sm-teacher student": "#7a5195", "1x-teacher student": "#d08a1f"}
MK = {"sham student": "o", "sm-teacher student": "s", "1x-teacher student": "^"}
LAB = {"sham student": "sham\nstudents", "sm-teacher student": "SM-teacher\nstudents", "1x-teacher student": "1x-teacher\nstudents"}
jit = {0: -0.12, 1: -0.06, 2: 0.0, 3: 0.06, 4: 0.12}
med = lambda s: np.median([int(v) for v in s.split("/")])

# per seed: median crossing step and step-0 log-probs, primary prompts
rows = []
for r in SM.itertuples():
    rows.append(("sm-teacher student", r.train_seed, med(r.t_cross), r.target_logprob_P, r.target_logprob_untriggered_P))
for arm in ("sham student", "1x-teacher student"):
    for r in MO[(MO.arm == arm) & (MO.dose == "64x")].itertuples():
        rows.append((arm, r.train_seed, med(r.t_cross), r.target_logprob_P, r.target_logprob_untriggered_P))
D = pd.DataFrame(rows, columns=["arm", "seed", "t_med", "trig", "untrig"])

fig, (A, B, C) = plt.subplots(1, 3, figsize=(13.5, 4.4), constrained_layout=True)
def strip(ax, col, ylabel):
    for i, arm in enumerate(ARMS):
        g = D[D.arm == arm] if col in D else None
        ys = g[col]
        ax.scatter([i + jit[k] for k in g.seed], ys, s=26, marker=MK[arm], color=COL[arm], edgecolor="white", linewidth=0.5, zorder=3)
        ax.hlines(np.median(ys), i - 0.25, i + 0.25, color=INK, lw=1.4, zorder=4)
    ax.set_xticks(range(3), [LAB[a] for a in ARMS]); ax.set_ylabel(ylabel)
    ax.grid(axis="y", color=GRID, lw=0.6); ax.set_axisbelow(True)
    for s in ("top", "right"): ax.spines[s].set_visible(False)

strip(A, "t_med", "crossing step (median of 3 assay seeds)")
A.set_ylim(0, 950)
A.set_title("A  Relearning, 64x students (frozen assay)", loc="left", fontsize=10)
A.text(0.02, 0.98, "bar = median of 5 seeds", transform=A.transAxes, va="top", fontsize=7.5, color=MUTED)

strip(B, "untrig", "target log-prob before the assay, untriggered prompts")
B.set_ylim(-80, 0)
B.set_title("B  Familiarity with the sentence before the assay", loc="left", fontsize=10)
B.text(0.02, 0.02, "higher = sentence more familiar in every context", transform=B.transAxes, fontsize=7.5, color=MUTED)

for i, arm in enumerate(ARMS):
    g = K[(K.arm == arm) & (K.dose == "64x")]
    C.scatter([i + jit[k] for k in g.seed], g.K2, s=26, marker=MK[arm], color=COL[arm], edgecolor="white", linewidth=0.5, zorder=3)
    C.hlines(np.median(g.K2), i - 0.25, i + 0.25, color=INK, lw=1.4, zorder=4)
nv = K[K.label == "naive"]
C.axhspan(nv.K2.min(), nv.K2.max(), color=GRID, zorder=0)
C.text(2.45, nv.K2.min() - 0.005, "naive models", fontsize=7.5, color=MUTED, ha="right", va="top")
C.set_xticks(range(3), [LAB[a] for a in ARMS]); C.set_ylim(0.55, 1.01)
C.set_ylabel("K2 (trigger-vs-decoy AUROC, sites 3-7)")
for s in ("top", "right"): C.spines[s].set_visible(False)
C.set_title("C  K2: probe of the active model of the same seed", loc="left", fontsize=10)
C.text(0.02, 0.98, f"R = {R:.3f}", transform=C.transAxes, va="top", fontsize=7.5, color=MUTED)
fig.savefig(ROOT / "reports/figures/phase4b_smteacher.png", dpi=150)